# Intermediate 06 — Authorization for MCP & Tool Servers

## Scenario

We operate an enterprise **Claims MCP Server**.

```text
Alice
  |
Claims Agent / MCP Client
  |
  | OAuth token: aud=claims-mcp
  v
Claims MCP
  |
  +-- claim.search
  +-- claim.read
  +-- claim.update
  +-- payment.create
  |
  v
Claims API / Payment API
```

We simulate the security controls locally so the lab runs without an IdP or MCP deployment, then map each control to production MCP/OAuth concepts.


In [ ]:
from datetime import datetime, timedelta, timezone
import copy, json, uuid
import jwt

ISSUER="https://id.example"
KEY="training-secret"

def now():
    return datetime.now(timezone.utc)

def issue_token(subject,audience,scopes,actor=None,ttl=300,extra=None):
    t=now()
    c={
      "iss":ISSUER,
      "sub":subject,
      "aud":audience,
      "scope":" ".join(sorted(scopes)),
      "iat":int(t.timestamp()),
      "exp":int((t+timedelta(seconds=ttl)).timestamp()),
      "jti":str(uuid.uuid4())
    }
    if actor:
        c["act"]={"sub":actor}
    if extra:
        c.update(extra)
    return jwt.encode(c,KEY,algorithm="HS256")


## 1 — Protected Resource Metadata

In [ ]:
protected_resource_metadata={
 "resource":"https://claims-mcp.example",
 "authorization_servers":["https://id.example"],
 "scopes_supported":[
   "claims:read",
   "claims:update",
   "payments:create"
 ],
 "bearer_methods_supported":["header"]
}
print(json.dumps(protected_resource_metadata,indent=2))


## 2 — Issue an MCP-specific token

In [ ]:
mcp_token=issue_token(
 subject="user:alice",
 actor="agent:claims-agent",
 audience="https://claims-mcp.example",
 scopes={"claims:read"},
 extra={
   "task_id":"task:483",
   "delegation_family":"dlg:483"
 }
)
print(jwt.decode(
 mcp_token,KEY,algorithms=["HS256"],
 audience="https://claims-mcp.example",
 issuer=ISSUER
))


## 3 — Validate token intent

In [ ]:
def validate_mcp_token(token,required_scope=None):
    c=jwt.decode(
      token,KEY,algorithms=["HS256"],
      audience="https://claims-mcp.example",
      issuer=ISSUER
    )
    scopes=set(c.get("scope","").split())
    if required_scope and required_scope not in scopes:
        raise PermissionError("insufficient_scope")
    return c

print(validate_mcp_token(mcp_token,"claims:read")["sub"])


## 4 — Audience confusion attack

In [ ]:
wrong_audience=issue_token(
 "user:alice",
 "https://calendar-api.example",
 {"claims:read"},
 actor="agent:claims-agent"
)

try:
    validate_mcp_token(wrong_audience)
except Exception as e:
    print("REJECTED:",type(e).__name__)


## 5 — Server, tool and resource policy

In [ ]:
TOOLS={
 "claim.search":{"scope":"claims:read","risk":"low"},
 "claim.read":{"scope":"claims:read","risk":"low"},
 "claim.update":{"scope":"claims:update","risk":"medium"},
 "payment.create":{"scope":"payments:create","risk":"high"},
}

AGENT_TOOLS={
 "agent:claims-agent":{"claim.search","claim.read","claim.update","payment.create"}
}

USER_RESOURCES={
 "user:alice":{"claim:483"}
}

TASK={
 "id":"task:483",
 "active":True,
 "resource":"claim:483",
 "allowed_tools":{"claim.search","claim.read","claim.update","payment.create"}
}


## 6 — Authorized tool discovery

In [ ]:
def visible_tools(token,task):
    c=validate_mcp_token(token)
    actor=c["act"]["sub"]
    scopes=set(c["scope"].split())
    visible=[]
    for name,meta in TOOLS.items():
        if name not in AGENT_TOOLS.get(actor,set()):
            continue
        if name not in task["allowed_tools"]:
            continue
        if meta["scope"] not in scopes:
            continue
        visible.append(name)
    return visible

print(visible_tools(mcp_token,TASK))


Discovery filtering improves least privilege and reduces capability disclosure, but it is **not** invocation enforcement. Every call must be checked again.

## 7 — Tool invocation authorization

In [ ]:
def authorize_tool(token,tool,task):
    if tool not in TOOLS:
        return False,"unknown tool"
    c=validate_mcp_token(token)
    actor=c["act"]["sub"]
    required=TOOLS[tool]["scope"]
    scopes=set(c["scope"].split())
    if required not in scopes:
        return False,"insufficient scope"
    if tool not in AGENT_TOOLS.get(actor,set()):
        return False,"agent not allowed tool"
    if not task["active"]:
        return False,"task inactive"
    if tool not in task["allowed_tools"]:
        return False,"tool outside task"
    return True,"tool authorized"

print(authorize_tool(mcp_token,"claim.read",TASK))
print(authorize_tool(mcp_token,"payment.create",TASK))


## 8 — Target-resource authorization

In [ ]:
def authorize_resource(token,resource_id,task):
    c=validate_mcp_token(token)
    user=f'user:{c["sub"].split(":")[-1]}'
    if resource_id not in USER_RESOURCES.get(user,set()):
        return False,"user cannot access resource"
    if resource_id != task["resource"]:
        return False,"resource outside task"
    return True,"resource authorized"

print(authorize_resource(mcp_token,"claim:483",TASK))
print(authorize_resource(mcp_token,"claim:999",TASK))


## 9 — End-to-end call

In [ ]:
def authorize_call(token,tool,args,task,approval=None):
    ok,reason=authorize_tool(token,tool,task)
    if not ok:
        return False,reason

    target=args.get("claim_id")
    if target:
        ok,reason=authorize_resource(token,target,task)
        if not ok:
            return False,reason

    if tool=="payment.create":
        if not approval or not approval["valid"]:
            return False,"step_up: approval required"
        if args["amount"]>approval["max_amount"]:
            return False,"amount exceeds approval"
        if args["claim_id"]!=approval["claim_id"]:
            return False,"approval bound to different claim"

    return True,"allow"

print(authorize_call(
 mcp_token,"claim.read",{"claim_id":"claim:483"},TASK
))


## 10 — Prompt-injected resource substitution

In [ ]:
print(authorize_call(
 mcp_token,
 "claim.read",
 {"claim_id":"claim:999"},
 TASK
))


## 11 — Step-up for sensitive tool

In [ ]:
print(authorize_call(
 mcp_token,
 "payment.create",
 {"claim_id":"claim:483","amount":300},
 TASK
))


## 12 — Scope accumulation

In [ ]:
old_claims=validate_mcp_token(mcp_token)
old_scopes=set(old_claims["scope"].split())
requested_additional={"payments:create"}

new_scopes=old_scopes | requested_additional
print("old:",old_scopes)
print("request during step-up:",new_scopes)

stepped_up=issue_token(
 "user:alice",
 "https://claims-mcp.example",
 new_scopes,
 actor="agent:claims-agent",
 extra={"task_id":"task:483","delegation_family":"dlg:483"}
)
print(validate_mcp_token(stepped_up)["scope"])


## 13 — Approval-bound payment

In [ ]:
approval={
 "id":"apr:92",
 "valid":True,
 "claim_id":"claim:483",
 "max_amount":500
}
print(authorize_call(
 stepped_up,
 "payment.create",
 {"claim_id":"claim:483","amount":300},
 TASK,
 approval
))
print(authorize_call(
 stepped_up,
 "payment.create",
 {"claim_id":"claim:483","amount":900},
 TASK,
 approval
))


## 14 — Confused deputy

In [ ]:
MCP_SERVICE_RIGHTS={"claim:*","payment:*"}

def unsafe_service_account_authorization(tool):
    # Anti-pattern: the MCP server's own broad rights decide user action.
    return tool in {"claim.read","claim.update","payment.create"}

print("unsafe:",unsafe_service_account_authorization("payment.create"))
print("safe:",authorize_call(
 mcp_token,
 "payment.create",
 {"claim_id":"claim:483","amount":300},
 TASK
))


## 15 — Reject token passthrough

In [ ]:
downstream_api="https://payment-api.example"

try:
    jwt.decode(
      stepped_up,KEY,algorithms=["HS256"],
      audience=downstream_api,
      issuer=ISSUER
    )
except Exception as e:
    print("Payment API rejects MCP token:",type(e).__name__)


## 16 — Downstream token exchange

In [ ]:
def exchange_for_downstream(mcp_token,target_audience,target_scopes):
    c=validate_mcp_token(mcp_token)
    source=set(c["scope"].split())
    mapping={
      "payments:create":{"payment:create"},
      "claims:read":{"claim:read"}
    }
    allowed=set()
    for s in source:
        allowed |= mapping.get(s,set())
    if not set(target_scopes)<=allowed:
        raise PermissionError("downstream privilege amplification")
    return issue_token(
      c["sub"],
      target_audience,
      set(target_scopes),
      actor=c["act"]["sub"],
      ttl=120,
      extra={"task_id":c["task_id"]}
    )

payment_token=exchange_for_downstream(
 stepped_up,
 "https://payment-api.example",
 {"payment:create"}
)
print(jwt.decode(
 payment_token,KEY,algorithms=["HS256"],
 audience="https://payment-api.example",
 issuer=ISSUER
)["scope"])


## 17 — Authorization-server issuer mix-up

In [ ]:
stored_auth_request={
 "expected_issuer":"https://id.example",
 "state":"state-123"
}

def finish_authorization(response_iss,state):
    if state != stored_auth_request["state"]:
        raise PermissionError("state mismatch")
    if response_iss != stored_auth_request["expected_issuer"]:
        raise PermissionError("authorization-server mix-up")
    return "code may be redeemed"

print(finish_authorization("https://id.example","state-123"))

try:
    finish_authorization("https://evil.example","state-123")
except PermissionError as e:
    print("REJECTED:",e)


## 18 — Issuer-bound client credentials

In [ ]:
CLIENT_REGISTRATIONS={
 "https://id.example":{
   "client_id":"claims-host-123",
   "issuer":"https://id.example"
 }
}

def credentials_for(issuer):
    c=CLIENT_REGISTRATIONS.get(issuer)
    if not c or c["issuer"]!=issuer:
        raise PermissionError("no issuer-bound registration")
    return c

print(credentials_for("https://id.example"))
try:
    credentials_for("https://other-id.example")
except PermissionError as e:
    print("REJECTED:",e)


## 19 — CIMD model

In [ ]:
client_id_metadata_document={
 "client_id":"https://agent-host.example/oauth/client-metadata.json",
 "client_name":"Enterprise Agent Host",
 "redirect_uris":["https://agent-host.example/oauth/callback"],
 "grant_types":["authorization_code"],
 "response_types":["code"]
}
print(json.dumps(client_id_metadata_document,indent=2))


This lab models the architectural idea. Use the exact MCP/OAuth metadata requirements of your SDK/spec revision in production.

## 20 — Enterprise-Managed Authorization

In [ ]:
ENTERPRISE_MCP_POLICY={
 "group:claims-team":{
   "servers":{"https://claims-mcp.example"},
   "agents":{"agent:claims-agent"},
   "max_scopes":{"claims:read","claims:update"}
 }
}

def enterprise_managed_access(group,server,agent,requested):
    p=ENTERPRISE_MCP_POLICY[group]
    return (
      server in p["servers"]
      and agent in p["agents"]
      and set(requested)<=p["max_scopes"]
    )

print(enterprise_managed_access(
 "group:claims-team",
 "https://claims-mcp.example",
 "agent:claims-agent",
 {"claims:read"}
))


## 21 — URL-mode downstream authorization model

In [ ]:
pending_downstream_auth={
 "request_id":"oauth:github:alice",
 "provider":"github",
 "authorization_url":"https://provider.example/oauth/authorize?...",
 "credential_destination":"server-side encrypted token store"
}
print(json.dumps(pending_downstream_auth,indent=2))


The security invariant is more important than this toy object:

```text
user authenticates to downstream provider in trusted browser flow
credential terminates at the authorized server-side integration
credential is not pasted into chat
credential is not exposed to the model
```


## 22 — Dynamic tool exposure

In [ ]:
def tools_for_risk(token,task,risk):
    tools=visible_tools(token,task)
    if risk>=80:
        return []
    if risk>=60:
        return [t for t in tools if TOOLS[t]["risk"]=="low"]
    return tools

print("low risk:",tools_for_risk(stepped_up,TASK,10))
print("high risk:",tools_for_risk(stepped_up,TASK,70))


## 23 — Cached discovery is not execution authority

In [ ]:
cached_tools=tools_for_risk(stepped_up,TASK,10)
print("cached catalog:",cached_tools)

# Risk changes after tools/list.
current_risk=85

def invocation_gate(token,tool,args,task,risk):
    if risk>=80:
        return False,"dynamic risk revoked tool authority"
    return authorize_call(token,tool,args,task,approval)

print(invocation_gate(
 stepped_up,
 "payment.create",
 {"claim_id":"claim:483","amount":300},
 TASK,
 current_risk
))


## 24 — Agent quarantine

In [ ]:
AGENT_STATUS={"agent:claims-agent":"active"}

def require_active_actor(token):
    c=validate_mcp_token(token)
    actor=c["act"]["sub"]
    if AGENT_STATUS.get(actor)!="active":
        raise PermissionError("agent quarantined")
    return c

require_active_actor(stepped_up)
AGENT_STATUS["agent:claims-agent"]="quarantined"

try:
    require_active_actor(stepped_up)
except PermissionError as e:
    print("REJECTED:",e)


## 25 — Audit evidence

In [ ]:
AUDIT=[]

def audited_call(token,tool,args,task,approval=None):
    c=validate_mcp_token(token)
    ok,reason=authorize_call(token,tool,args,task,approval)
    record={
      "decision_id":str(uuid.uuid4()),
      "subject":c["sub"],
      "actor":c["act"]["sub"],
      "mcp_server":"https://claims-mcp.example",
      "tool":tool,
      "target":args.get("claim_id"),
      "task":task["id"],
      "decision":"allow" if ok else "deny",
      "reason":reason,
      "token_jti":c["jti"]
    }
    AUDIT.append(record)
    return record

print(json.dumps(audited_call(
 stepped_up,
 "payment.create",
 {"claim_id":"claim:483","amount":300},
 TASK,
 approval
),indent=2))


## 26 — Adversarial regression suite

In [ ]:
# Reset agent for tests.
AGENT_STATUS["agent:claims-agent"]="active"

tests=[
 ("wrong resource",
  lambda: authorize_call(mcp_token,"claim.read",{"claim_id":"claim:999"},TASK)[0],
  False),
 ("missing payment scope",
  lambda: authorize_call(mcp_token,"payment.create",{"claim_id":"claim:483","amount":300},TASK,approval)[0],
  False),
 ("amount escalation",
  lambda: authorize_call(stepped_up,"payment.create",{"claim_id":"claim:483","amount":900},TASK,approval)[0],
  False),
 ("valid read",
  lambda: authorize_call(mcp_token,"claim.read",{"claim_id":"claim:483"},TASK)[0],
  True),
]
for name,fn,expected in tests:
    got=fn()
    print(name,got)
    assert got==expected


## 27 — Exercise: real MCP Python SDK

Using the current MCP Python SDK:

1. create a remote HTTP MCP server;
2. publish Protected Resource Metadata;
3. configure an OAuth authorization server;
4. validate bearer tokens;
5. expose `claim.read` and `claim.update`;
6. return correct HTTP 401/403 challenges;
7. verify the token is intended for the MCP resource.

Do not use the notebook's training JWT secret in a real deployment.


## 28 — Exercise: OPA policy

The course includes:

```text
policies/opa/mcp.rego
```

Move these checks into OPA:

```text
agent permission
task binding
target resource
amount
approval
risk
```

Keep cryptographic token validation at the appropriate trusted boundary and pass verified claims to policy.


## 29 — Exercise: Cedar

The course includes:

```text
policies/cedar/mcp.cedar
```

Model:

```text
Agent
User
Tool
Claim
Task
```

Then compare Cedar's typed PARC model with the Rego approach.


## 30 — Exercise: downstream GitHub authorization

Design an MCP server that needs GitHub access.

Compare:

```text
A. unsafe token passthrough
B. OAuth token exchange/OBO where supported
C. server-managed downstream OAuth via URL-mode elicitation
```

Document where each credential terminates and which audience receives it.


## 31 — Exercise: Enterprise-Managed Authorization

Design EMA policy for:

```text
Claims department
Finance department
Engineering
```

Each department gets a centrally approved MCP catalog, but fine-grained tool/resource authorization still happens inside each MCP service.

Explain why EMA does not replace application authorization.


## 32 — Exercise: gateway enforcement

The 2026-07-28 protocol can expose method/tool names in HTTP headers.

Design gateway controls for:

```text
rate limits
coarse tool deny list
server allow list
tenant routing
observability
```

Then explain which checks still must remain inside the MCP service.


## 33 — Review questions

1. What role does an MCP server play in OAuth?
2. What is Protected Resource Metadata?
3. Why must an MCP server validate token intent/audience?
4. What are OAuth Resource Indicators?
5. Why is token passthrough dangerous?
6. What is a confused deputy?
7. What are the three authorization layers in this course?
8. Why is tool discovery part of the security design?
9. Why are tool annotations not authorization?
10. Why preserve user and agent identity separately?
11. Why are tool arguments authorization inputs?
12. What does step-up mean for MCP?
13. Why should prior scopes be preserved during appropriate scope step-up?
14. What OAuth mix-up risk does `iss` validation address?
15. Why must client credentials be isolated by authorization-server issuer?
16. What is CIMD and why is MCP moving toward it?
17. What problem does Enterprise-Managed Authorization solve?
18. Why does EMA not replace fine-grained tool/resource authorization?
19. How does URL-mode elicitation help downstream OAuth?
20. Why is a cached `tools/list` response not execution authority?
21. What does stateless MCP change about authorization design?
22. Why must long-running MCP tasks be re-authorized?
23. Why should tool implementation/version changes be governed?
24. What evidence should be recorded for sensitive tool calls?

# Next course

## Intermediate 07 — Risk, Assurance & Step-Up Authorization for Agents
